In [ ]:
from utils.TimeSeriesProcessing import *
import numpy as np
import pandas as pd
import os
import shutil
import matplotlib.pyplot as plt
import xlrd
import openpyxl

In [ ]:
# def copy_formatting(source_sheet, target_sheet):
#     for row in range(source_sheet.nrows):
#         for col in range(source_sheet.ncols):
#             cell = source_sheet.cell(row, col)
#             new_cell = target_sheet.cell(row=row + 1, column=col + 1, value=cell.value)

#             if cell.xf_index:
#                 xf = source_sheet.book.xf_list[cell.xf_index]
#                 bgx = xf.background.pattern_colour_index
#                 color = source_sheet.book.colour_map.get(bgx)

#                 # Convert RGB tuple to hex string if color is available
#                 if color:
#                     # Format RGB tuple to hex string (e.g., 'FF112233')
#                     rgb_hex = ''.join(f"{value:02X}" for value in color)
#                     new_color = openpyxl.styles.colors.Color(rgb=rgb_hex)
#                     fill = openpyxl.styles.fills.PatternFill(patternType='solid', fgColor=new_color)
#                     new_cell.fill = fill

# directory_path = "path/to/raw_study_data"
# destination_path = "path/to/ctg_recordings"
# os.makedirs(destination_path, exist_ok=True)

# for filename in os.listdir(directory_path):
#     if filename.endswith(".xls"):
#         source_path = os.path.join(directory_path, filename)
#         destination_file = filename.replace(".xls", ".xlsx")
#         output_path = os.path.join(destination_path, destination_file)

#         # Read the original xls file
#         wb = xlrd.open_workbook(source_path, formatting_info=True)
#         sheet = wb.sheet_by_index(0)

#         # Create a new xlsx file
#         new_wb = openpyxl.Workbook()
#         new_sheet = new_wb.active

#         # Copy data and formatting
#         copy_formatting(sheet, new_sheet)

#         # Save the new xlsx file
#         new_wb.save(output_path)

# print("Conversion to .xlsx completed for all files.")

In [ ]:
source_directory = "path/to/raw_study_data"

# Destination directory where .xlsx files will be moved
destination_directory = "../updatedCTGxlxs"

# Create the destination directory if it does not exist
os.makedirs(destination_directory, exist_ok=True)

# Iterate over each file in the source directory
for filename in os.listdir(source_directory):
    if filename.endswith(".xlsx"):  # Check for .xlsx files
        source_path = os.path.join(source_directory, filename)
        destination_path = os.path.join(destination_directory, filename)
        # Move the file
        shutil.move(source_path, destination_path)

print("All .xlsx files have been moved to the new directory.")

In [ ]:
dir = "path/to/ctg_recordings"
# Load the data
color_code = {}
for filename in os.listdir(dir):
    if not filename.endswith(".xlsx"):
        continue
    print(f"Processing file: {filename}")
    color = getCellColor(os.path.join(dir, filename), row=4, col=1)
    color_code[filename] = color
color_code

In [ ]:
dir = "path/to/ctg_recordings"
data_list = prepData(dir)

In [ ]:
file_path = "./data_list.npz"
data_list = []
with np.load(file_path, allow_pickle=True) as data:
    data = data['arr_0']
    data_list.extend(data)

In [ ]:
Study_IDs = []
gender = []
Primigravida = []
Meconium = []
Delivery_Type = []
GA_weeks = []
Birth_Weight = []
label = []
for data in data_list:
    Study_IDs.append(data['Study_ID'])
    gender.append(data['Gender'])
    Primigravida.append(data['Primigravida'])
    Meconium.append(data['Meconium'])
    Delivery_Type.append(data['Delivery_Type'])
    GA_weeks.append(data['GA'])
    Birth_Weight.append(data['Birthweight'])
    label.append(data['label'] if 'label' in data else None)  # Handle cases where 'label' might not exist
# Create a DataFrame from the lists
df = pd.DataFrame()
df['Study_ID'] = Study_IDs
df['gender'] = gender
df['Primigravida'] = Primigravida
df['Meconium'] = Meconium
df['Delivery_Type'] = Delivery_Type
df['GA_weeks'] = GA_weeks
df['Birth_Weight'] = Birth_Weight
df['label'] = label

df.head()

In [ ]:
# demographic information
df['label'].value_counts()  # Check the distribution of labels in the dataset

In [ ]:
# calculate the sparsity distribution of the time-series sequence
sparsity_list = []
for data in data_list:
    print(data["Study_ID"])
    measure = pd.DataFrame(data['df'])
    print(type(measure))
    print(measure.columns)
    fhr_data = measure['FHR1'].values
    zero_input = np.count_nonzero(fhr_data == 0)
    sparsity = zero_input / fhr_data.shape[0]
    sparsity_list.append(sparsity)

In [ ]:
sparsity_array = np.array(sparsity_list)
plt.hist(sparsity_array, bins=20)
plt.xlabel('Sparsity')

In [ ]:
# # Loop through every subject in data_list, get labels and "df[FHR1]"
# # and crop the last 30 minutes of data (length = 30*60*4) of the "df[FHR1]".
# # Then stack the signals and labels of each subject and save to npz file.

# def process_data(data_list, length):
#     data = []
#     labels = []
#     for subject in data_list:
#         df = subject["df"]
        
#         # Check if 'FHR1' is in the DataFrame, otherwise use the second column
#         if 'FHR1' in df.columns:
#             fhr_data_column = df['FHR1']
#         else:
#             # Access the second column, ensuring the DataFrame has at least two columns
#             if df.shape[1] < 2:
#                 print("DataFrame has less than two columns, skipping subject.")
#                 continue
#             fhr_data_column = df.iloc[:, 1]  # Second column
        
#         # Identify the last non-zero data point in the selected column
#         last_non_zero_index = fhr_data_column[fhr_data_column != 0].last_valid_index()
#         if last_non_zero_index is None or last_non_zero_index < length - 1:
#             print(f"Skipping subject with Study_ID {subject['Study_ID']} due to insufficient non-zero data points.")
#             # Skip this subject if there aren't enough data points
#             continue
        
#         # Calculate the start index for slicing
#         start_index = max(0, last_non_zero_index - length + 1)
        
#         # Slice the data to get the last 'length' entries from the last non-zero point
#         fhr_data = fhr_data_column[start_index:last_non_zero_index + 1]
        
#         # Check if we have enough data, if not, continue to next subject
#         if len(fhr_data) < length:
#             continue
        
#         labels.append(subject["label"])
#         data.append(fhr_data)
        
#     return np.array(data, dtype=object), np.array(labels)

# # Example call to the function
# data, labels = process_data(data_list, 30*60*4)
# np.savez("../FHR30minFull", X=data, y=labels)
    

In [ ]:
from collections import defaultdict
import numpy as np

def segment_trace(
    trace,
    label,
    subject_id,
    segment_length=30 * 60 * 4,
    pad_last=True,
    fhr_baseline=130.0,
    return_mask=True,
    min_observed_ratio=0.3,
    stats=None,  # <-- NEW
):
    """
    Segment FHR trace with NaN handling, missingness mask,
    and drop overly sparse segments. Logs stats if provided.
    """
    trace = trace.copy()
    T = trace.shape[-1]

    # ---- 1. Missingness mask (1 = observed, 0 = missing) ----
    mask = (~np.isnan(trace)).astype(np.float32)

    # ---- 2. Fill NaNs ----
    trace = np.nan_to_num(trace, nan=fhr_baseline)

    segments = []
    num_full = T // segment_length

    def keep_segment(seg_mask):
        observed_ratio = seg_mask.mean()
        return observed_ratio >= min_observed_ratio

    # ---- 3. Full segments ----
    for i in range(num_full):
        start = i * segment_length
        end = start + segment_length

        seg_trace = trace[..., start:end]
        seg_mask = mask[..., start:end]

        if stats is not None:
            stats["before"][label] += 1

        if not keep_segment(seg_mask):
            if stats is not None:
                stats["dropped"][label] += 1
            continue

        if stats is not None:
            stats["after"][label] += 1

        if return_mask:
            segments.append((seg_trace, seg_mask, label, subject_id))
        else:
            segments.append((seg_trace, label, subject_id))

    # ---- 4. Last (padded) segment ----
    remainder = T % segment_length
    if remainder > 0 and pad_last:
        pad_width = segment_length - remainder

        pad_shape = list(trace.shape)
        pad_shape[-1] = pad_width

        trace_pad = np.full(pad_shape, fhr_baseline, dtype=trace.dtype)
        mask_pad = np.zeros(pad_shape, dtype=np.float32)

        trace_last = np.concatenate(
            [trace[..., num_full * segment_length:], trace_pad],
            axis=-1
        )
        mask_last = np.concatenate(
            [mask[..., num_full * segment_length:], mask_pad],
            axis=-1
        )

        if stats is not None:
            stats["before"][label] += 1

        if keep_segment(mask_last):
            if stats is not None:
                stats["after"][label] += 1

            if return_mask:
                segments.append((trace_last, mask_last, label, subject_id))
            else:
                segments.append((trace_last, label, subject_id))
        else:
            if stats is not None:
                stats["dropped"][label] += 1

    return segments

In [ ]:
fhr_traces = [data['df']['FHR1'].values for data in data_list]
toco_traces = [data['df']['TOCO'].values for data in data_list]
labels = [data['label'] for data in data_list]
subject_ids = [data['Study_ID'] for data in data_list]

In [ ]:
import numpy as np

# segment FHR and TOCO traces into fixed-length CTG patches with hybrid normalization
def segment_ctg_trace(
    fhr,
    toco,
    label,
    subject_id,
    segment_length=30 * 60 * 4,  # 30 min × 60 sec × 4 Hz
    pad_last=True,
    eps=1e-6
):
    """
    Segment FHR and TOCO into fixed-length CTG patches with
    hybrid normalization:
        - instance-normalized signals
        - per-segment FHR baseline statistics

    Output channels:
        0: FHR_norm
        1: TOCO_norm
        2: FHR_mean (constant)
        3: FHR_std  (constant)
    """

    # ----------------------
    # align length
    # ----------------------
    T = min(len(fhr), len(toco))
    fhr = fhr[:T].astype(np.float32)
    toco = toco[:T].astype(np.float32)

    trace = np.stack([fhr, toco], axis=0)  # (2, T)

    segments = []
    num_full = T // segment_length

    # ----------------------
    # helper: hybrid encoding
    # ----------------------
    def hybrid_encode(fhr_seg, toco_seg):
        # instance normalization
        fhr_mean = fhr_seg.mean()
        fhr_std = fhr_seg.std()
        toco_mean = toco_seg.mean()
        toco_std = toco_seg.std()

        fhr_norm = (fhr_seg - fhr_mean) / (fhr_std + eps)
        toco_norm = (toco_seg - toco_mean) / (toco_std + eps)

        L = fhr_seg.shape[0]

        # broadcast baseline stats
        fhr_mean_chan = np.full(L, fhr_mean, dtype=np.float32)
        fhr_std_chan = np.full(L, fhr_std, dtype=np.float32)

        return np.stack(
            [fhr_norm, toco_norm, fhr_mean_chan, fhr_std_chan],
            axis=0
        )  # (4, L)

    # ----------------------
    # full segments
    # ----------------------
    for i in range(num_full):
        start = i * segment_length
        end = start + segment_length

        fhr_seg = fhr[start:end]
        toco_seg = toco[start:end]

        seg = hybrid_encode(fhr_seg, toco_seg)
        print(f"Segment {i}: seg shape={seg.shape}, label={label}, subject_id={subject_id}")
        segments.append((seg, label, subject_id))

    # ----------------------
    # leftover segment
    # ----------------------
    remainder = T % segment_length
    if remainder > 0 and pad_last:
        fhr_seg = fhr[num_full * segment_length:]
        toco_seg = toco[num_full * segment_length:]

        seg = hybrid_encode(fhr_seg, toco_seg)

        pad_width = segment_length - remainder
        padding = np.zeros((seg.shape[0], pad_width), dtype=np.float32)

        seg = np.concatenate([seg, padding], axis=1)
        segments.append((seg, label, subject_id))

    return segments

In [ ]:
ctg_segments = []
for fhr, toco, label, subject_id in zip(fhr_traces, toco_traces, labels, subject_ids):
    segments = segment_ctg_trace(fhr, toco, label, subject_id, segment_length=10*60*4, pad_last=True)
    ctg_segments.extend(segments)
# inspect the number of segments and corresponding label counts
print(f"Total segments: {len(ctg_segments)}")
print(f"Dimension of a segment: {ctg_segments[0][0].shape}")
print(f"Subject ID of first segment: {ctg_segments[0][2]}")
print(f"Label counts: {np.unique([label for _, label, _ in ctg_segments], return_counts=True)}")

In [ ]:
from collections import defaultdict

stats = {
    "before": defaultdict(int),
    "after": defaultdict(int),
    "dropped": defaultdict(int),
}

fhr_segments = []
for fhr, label, subject_id in zip(fhr_traces, labels, subject_ids):
    fhr = fhr.astype(np.float32)
    segments = segment_trace(fhr, label, subject_id, segment_length=20*60*4, pad_last=True, fhr_baseline=130.0, return_mask=True, min_observed_ratio=0.5, stats=stats)
    fhr_segments.extend(segments)

print(f"Total segments: {len(fhr_segments)}")
print(f"Dimension of a segment: {fhr_segments[0][0].shape}")
print(f"Subject ID of first segment: {fhr_segments[0][3]}")
print(f"Label counts: {np.unique([label for _, _, label, _ in fhr_segments], return_counts=True)}")
print(f"Stats: {stats}")

In [ ]:
# save univariate FHR segments
fhr_segments = []
for fhr, label, subject_id in zip(fhr_traces, labels, subject_ids):
    fhr = fhr.astype(np.float32)
    segments = segment_trace(fhr, label, subject_id, segment_length=20*60*4, pad_last=True, fhr_baseline=130.0, return_mask=True, min_observed_ratio=0.5)
    fhr_segments.extend(segments)
# inspect the number of segments and corresponding label counts
print(f"Total segments: {len(fhr_segments)}")
print(f"Dimension of a segment: {fhr_segments[0][0].shape}")
print(f"Missingness mask of first segment: {fhr_segments[0][1]}")
print(f"Subject ID of first segment: {fhr_segments[0][-1]}")
# print(f"Label counts: {np.unique([label for _, label, _ in fhr_segments], return_counts=True)}")

In [ ]:
X = np.stack([s for s, _, _, _ in fhr_segments])
y = np.array([l for _, _, l, _ in fhr_segments])
subject_ids = np.array([sid for _, _, _, sid in fhr_segments])
print(f"Final dataset shape: X={X.shape}, y={y.shape}, subject_ids={subject_ids.shape}")

In [ ]:
X = np.stack([s for s, _, _ in ctg_segments])
y = np.array([l for _, l, _ in ctg_segments])
subject_ids = np.array([sid for _, _, sid in ctg_segments])
print(f"Final dataset shape: X={X.shape}, y={y.shape}, subject_ids={subject_ids.shape}")

In [ ]:
np.savez_compressed(
    "./dataset/FHR20minSegments05missing.npz",
    X=X.astype(np.float32),
    y=y.astype(np.int64),
    subject_ids=subject_ids,
    channels=np.array(["FHR"]),
    fs=4,
    window_minutes=20
)

In [ ]:
np.savez_compressed(
    "./dataset/CTG10minSegments4Chan.npz",
    X=X.astype(np.float32),
    y=y.astype(np.int64),
    subject_ids=subject_ids,
    channels=np.array(["FHR", "TOCO", "FHR_mean", "FHR_std"]),
    fs=4,
    window_minutes=10
)

In [ ]:
data = np.load("dataset/CTG10minSegments4Chan.npz")

subject_ids = data["subject_ids"]
labels = data["y"]

print("subject_ids shape:", subject_ids.shape)
print("labels shape:", labels.shape)

print("Unique subject_ids:", np.unique(subject_ids))
print("Unique labels:", np.unique(labels))